# Smart Agent: LoRA fine-tune of Llama 3.2 3B (Unsloth, free Colab T4)

Run the cells top to bottom. **Runtime > Change runtime type > T4 GPU** first.
Needs `all_train.jsonl` and `all_eval.jsonl` from the project's `training/` folder (first run only; they are then kept in Drive).

**If Colab disconnects:** reconnect, then re-run every cell from the top. The training cell resumes from the last
checkpoint saved in Drive (every 100 steps), so you lose at most 100 steps.

In [ ]:
!nvidia-smi -L
!pip install -q unsloth

## 1. Connect Google Drive and get the data files
Checkpoints and data are kept in Drive, so if Colab disconnects you only re-run the cells from the top
(they pick up where training stopped). Click "Connect" when the Drive pop-up appears.

In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/smart-agent"
os.makedirs(DRIVE, exist_ok=True)

need = ["all_train.jsonl", "all_eval.jsonl"]
if all(os.path.exists(f"{DRIVE}/{f}") for f in need):
    for f in need:
        shutil.copy(f"{DRIVE}/{f}", f)
    print("loaded data from Drive")
else:
    from google.colab import files
    files.upload()   # pick all_train.jsonl and all_eval.jsonl
    for f in need:
        shutil.copy(f, f"{DRIVE}/{f}")
    print("uploaded and copied to Drive")

## 2. Load the base model (4-bit) and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Prepare the data
Rows with two tool calls in one message are split into one call at a time, because the Llama chat
template only supports a single call per turn (the agent loop handles one call per round anyway).

In [ ]:
import json
from datasets import Dataset

def split_calls(msgs):
    out, i = [], 0
    while i < len(msgs):
        m = msgs[i]
        calls = m.get("tool_calls")
        if m["role"] == "assistant" and calls and len(calls) > 1:
            n = len(calls)
            for c, r in zip(calls, msgs[i + 1:i + 1 + n]):
                out.append({"role": "assistant", "content": "", "tool_calls": [c]})
                out.append(r)
            i += 1 + n
        else:
            out.append(m)
            i += 1
    return out

def load(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    for r in rows:
        r["messages"] = split_calls(r["messages"])
    return rows

train_rows, eval_rows = load("all_train.jsonl"), load("all_eval.jsonl")

def render(r):
    return tokenizer.apply_chat_template(r["messages"], tools=r.get("tools"), tokenize=False)

train_ds = Dataset.from_dict({"text": [render(r) for r in train_rows]})
lens = [len(tokenizer(t).input_ids) for t in train_ds["text"][:200]]
print(len(train_ds), "rows; max tokens in first 200:", max(lens))   # must stay under 2048
print(train_ds[0]["text"][-700:])

## 4. Train (about 45-70 minutes on a free T4; lower `num_train_epochs` to 2 to go faster)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        seed=3407,
        output_dir=f"{DRIVE}/checkpoints",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
    ),
)
# learn only from assistant turns (not the system prompt, tool schema, or tool results)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
import glob
resume = bool(glob.glob(f"{DRIVE}/checkpoints/checkpoint-*"))
print("resuming from the latest checkpoint" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=resume)

## 5. Test on the held-out rows (these were never trained on)
Checks the **first** decision of each conversation: the right tool and arguments, or no tool when none is needed.

In [ ]:
import re, random
FastLanguageModel.for_inference(model)

def parse_call(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    if "name" not in d:
        return None
    args = d.get("parameters", d.get("arguments", {})) or {}
    return d["name"], {k: str(v) for k, v in args.items()}

def first_assistant(msgs):
    return next(i for i, m in enumerate(msgs) if m["role"] == "assistant")

N = 80   # use len(eval_rows) for the full set
random.seed(1)
sample = random.sample(eval_rows, min(N, len(eval_rows)))
stats = {"call": [0, 0], "nocall": [0, 0]}
fails = []
for r in sample:
    msgs = r["messages"]
    i = first_assistant(msgs)
    prompt = tokenizer.apply_chat_template(msgs[:i], tools=r.get("tools"), add_generation_prompt=True, tokenize=False)
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=150, do_sample=False)
    gen = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    exp = msgs[i]
    got = parse_call(gen)
    if exp.get("tool_calls"):
        f = exp["tool_calls"][0]["function"]
        want = (f["name"], {k: str(v) for k, v in f["arguments"].items()})
        kind, good = "call", got == want
    else:
        kind, good = "nocall", got is None
    stats[kind][0] += good
    stats[kind][1] += 1
    if not good:
        fails.append((msgs[i - 1]["content"], exp.get("tool_calls") or exp["content"], gen))

for k, (a, b) in stats.items():
    print(f"{k}: {a}/{b}" + (f" = {100*a/b:.0f}%" if b else ""))
for u, want, got in fails[:10]:
    print("\nUSER:", u, "\n WANT:", want, "\n GOT :", got)

## 6. Try it live
Edit the question, run the cell. (No tools are offered here, so this tests knowledge/chat, not tool calls.)

In [ ]:
def ask(q):
    msgs = [{"role": "system", "content": "You are a voice assistant. Keep replies to one or two short sentences."},
            {"role": "user", "content": q}]
    p = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(p, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for q in ["What should you call me?", "What grade did I get in operating systems?",
          "What is the capital of Iran?", "What is Cofferant?"]:
    print(q, "->", ask(q))

## 7. Save to Google Drive (adapters + GGUF for Ollama)
Only run this after the test above looks good. The GGUF is about 2 GB.

In [ ]:
import glob, shutil, os
dest = DRIVE   # Drive was mounted in step 1

model.save_pretrained("lora_adapters")              # small, lets you retrain/merge later
tokenizer.save_pretrained("lora_adapters")
shutil.copytree("lora_adapters", f"{dest}/lora_adapters", dirs_exist_ok=True)

model.save_pretrained_gguf("smart_llama", tokenizer, quantization_method="q4_k_m")
g = glob.glob("smart_llama*/*.gguf") or glob.glob("**/*Q4_K_M*.gguf", recursive=True)
print(g)
shutil.copy(g[0], f"{dest}/smart-llama-q4_k_m.gguf")
print("saved to Drive:", dest)